Ce markdown contient toutes les étapes de nétoyages / corrections / Merges sur les données utilisées

In [ ]:
# Importation des modules
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats as stats  # noqa: PLR0402


In [ ]:
# Importation de la base Navco (En commentaire car les bases sont déjà spécifiées dans le Main)
# navco = pd.read_excel("Data/base2-1.xls")
# navco.head()
# navco.shape
corrected = False

In [ ]:
# Importation de la base "population" (En commentaire car les bases sont déjà spécifiées dans le Main)
# pop = pd.read_csv("Data/population.csv", sep=",")
pop = pop.rename(columns={"Entity": "location", "Year":"year"})

# pop.shape

In [ ]:
# Correction des noms de pays pour merge correctement (attention à ne run qu'une fois, ou à recharger les données)

if corrected == False:
    corrections = {
        "Myanmar/Burma": "Myanmar",
        "Palestinian Territories": "Palestine",
        "UK": "United Kingdom",
        "Cambodia (Kampuchea)": "Cambodia",
        "Bosnia-Herzegovina": "Bosnia and Herzegovina",
        "Ivory Coast": "Cote d'Ivoire",
        "Swaziland": "Eswatini",
        "Macedonia": "North Macedonia",
        "Hyderabad": "India",
        "Princely State of Travencore": "India",
        "Yemen (North Yemen)" : "Yemen Arab Republic"
        #pays absents de la base population 
        #"South Vietnam": "Vietnam", 
        #"Tibet": "China",
        
    }
    navco["location"] = navco["location"].replace(corrections)
    navco.loc[(navco["location"] == "Yugoslavia") & (navco["year"] > 1992), "location"] = "Serbia and Montenegro"

    nouvelles_lignes = pd.DataFrame(
        [{"location": "East Germany", "year": 1953, "Population": 18_390_000},#date de 1950 dans la base
        {"location": "East Germany", "year": 1989, "Population": 16_110_000}]#date de 1990
        + [{"location": "Yemen Arab Republic", "year": a, "Population": 4_880_000} for a in range(1962, 1970)] #date de 1970 (pas de surestimation de la proportion de la population)
    )
    pop = pd.concat([pop, nouvelles_lignes], ignore_index=True)

    navco["location"] = navco["location"].str.strip() #retirer les éventuels espaces avant ou après

    corrected = True

In [33]:
# Merge de base_1 et de population
navco_pop = navco.merge(
    pop[["location", "year", "Population"]],
    on=["location", "year"],
    how="left",                 
    validate="many_to_one",
    indicator=True
)

print(len(navco), len(navco_pop)) #vérification du nombre de lignes            
print(navco_pop["_merge"].value_counts())
manquants = (
    navco_pop.loc[navco_pop["_merge"] == "left_only", ["location", "year"]]
    .drop_duplicates()
    .sort_values(["location", "year"])
)
print(manquants.to_string(index=False))
 #regarder les pays qui n'ont pas pu fusionner avec les dates

navco_pop["total_part_clean"] = navco_pop["total_part"].replace(-99, np.nan) #enlever valeurs manquantes de la participation tot
navco_pop[["location", "year", "Population"]].head(20)

2717 2717
_merge
both          2699
left_only       18
right_only       0
Name: count, dtype: int64
     location  year
South Vietnam  1958
South Vietnam  1959
South Vietnam  1960
South Vietnam  1961
South Vietnam  1962
South Vietnam  1963
South Vietnam  1964
South Vietnam  1965
South Vietnam  1966
        Tibet  1956
        Tibet  1957
        Tibet  1958
        Tibet  1959
        Tibet  1987
        Tibet  1988
        Tibet  1989


,location,year,Population
0,Madagascar,1991,12421413.0
1,Madagascar,1992,12800870.0
2,Madagascar,1993,13197353.0
3,Djibouti,1991,631096.0
4,Djibouti,1992,626016.0
5,Djibouti,1993,614247.0
6,Djibouti,1994,626717.0
7,Afghanistan,1979,13655575.0
8,Afghanistan,1980,13169313.0
9,Afghanistan,1981,11937586.0


In [ ]:
navco_pop["total_part_clean"] = navco_pop["total_part"].replace(-99, np.nan) #enlever valeurs manquantes de la participation tot

navco_pop["pourcent_part"] = (navco_pop["total_part_clean"] / navco_pop["Population"]) * 100 #calcul pourcentage de pop dans mouvement

print(navco_pop[["location", "year", "total_part_clean", "Population", "pourcent_part"]].head(10))

print(navco_pop.loc[navco_pop["pourcent_part"] > 20, ["id","location","camp_name", "year","total_part_clean","pourcent_part","Population"]])
#regarder les valeurs aberrantes 

masque_50 = navco_pop["pourcent_part"] > 50 #après vérification des sources, toutes les valeurs sup à 50 % sont issues d'erreurs dans la base 2-1
navco_pop.loc[masque_50, ["total_part_clean", "pourcent_part"]] = np.nan #on remplace ces valeurs par des valeurs manquantes


df_pc_pop = navco_pop.dropna(subset=["pourcent_part", "progress"]).copy() #enlever valeurs manquantes dans progress et pourcent

recodage = {5: 0, 0: 1, 1: 2, 2: 3, 3: 4, 4: 5} #recodage car dans la variable progress, 5 : failure
df_pc_pop["progress2"] = df_pc_pop["progress"].map(recodage)

      location  year  total_part_clean  Population  pourcent_part
0   Madagascar  1991          400000.0  12421413.0       3.220246
1   Madagascar  1992               NaN  12800870.0            NaN
2   Madagascar  1993               NaN  13197353.0            NaN
3     Djibouti  1991            3000.0    631096.0       0.475363
4     Djibouti  1992            4500.0    626016.0       0.718831
5     Djibouti  1993            4500.0    614247.0       0.732604
6     Djibouti  1994            4500.0    626717.0       0.718027
7  Afghanistan  1979           15000.0  13655575.0       0.109845
8  Afghanistan  1980           44500.0  13169313.0       0.337907
9  Afghanistan  1981           45500.0  11937586.0       0.381149
       id        location                            camp_name  year  \
63     24          Panama                         Anti-Noriega  1987   
64     24          Panama                         Anti-Noriega  1988   
65     24          Panama                         Anti-Nor

In [ ]:
# Importation de la base Navco 2 (Nouvelle itération pour éviter les conflits) (En commentaire car les )
# navco = pd.read_excel("Data/base2-1.xls")
navco2 = navco.copy()
num_cols = navco2.select_dtypes("number").columns
navco2[num_cols] = navco2[num_cols].replace(-99, np.nan)
navco2["violente"] = (navco2["prim_meth"] == 0).astype(int)
base = navco2.sort_values(["id", "year"]).reset_index(drop=True)